In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(dirname, "→", len(filenames), "files")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta → 2 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio → 2000 files
/kaggle/input/jan-2026-dl-ge

In [2]:
# ===== W&B INSTALL (Kaggle) =====
!pip install --upgrade wandb -q
# ===== W&B LOGIN =====
import wandb
wandb.login(key="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")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 75.5 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import os
import random
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score

# =========================
# CONFIG
# =========================

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

STEMS_PATH = f"{BASE_PATH}/genres_stems"
TEST_PATH = f"{BASE_PATH}/mashups"
SUB_PATH = f"{BASE_PATH}/sample_submission.csv"

SR = 22050
DURATION = 8
SAMPLES = SR * DURATION

N_MELS = 256

TRAIN_MULTIPLIER = 16

BATCH_SIZE = 48
EPOCHS = 25
LR = 4e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# ===== W&B INIT =====
wandb.init(
    project="dl-genAI-2026",
    name="efficientnet_b0_run",
    
    config={
        "sample_rate": SR,
        "duration": DURATION,
        "n_mels": N_MELS,
        "train_multiplier": TRAIN_MULTIPLIER,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "model": "efficientnet_b0",
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
    }
)

config = wandb.config

In [5]:
songs = []
labels = []

genres = sorted(os.listdir(STEMS_PATH))
label_to_idx = {g: i for i, g in enumerate(genres)}
idx_to_label = {v: k for k, v in label_to_idx.items()}

for genre in genres:

    genre_path = os.path.join(STEMS_PATH, genre)

    for song_folder in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song_folder)

        stems = {
            "drums": os.path.join(song_path, "drums.wav"),
            "bass": os.path.join(song_path, "bass.wav"),
            "vocals": os.path.join(song_path, "vocals.wav"),
            "other": os.path.join(song_path, "other.wav")
        }

        songs.append(stems)
        labels.append(label_to_idx[genre])

In [6]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)

for train_idx, val_idx in sss.split(songs, labels):

    train_songs = [songs[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]

    val_songs = [songs[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]

In [7]:
def random_crop(y):

    if len(y) <= SAMPLES:
        return np.pad(y, (0, SAMPLES - len(y)))

    start = np.random.randint(0, len(y) - SAMPLES)
    return y[start:start + SAMPLES]


def load_audio(path):

    y, _ = librosa.load(path, sr=SR, mono=True)
    return random_crop(y)


STEMS = ["drums", "bass", "vocals", "other"]

def generate_mixes():

    mixes = [

        ["drums","bass","vocals","other"],

        ["drums","bass"],
        ["drums","vocals"],
        ["drums","other"],

        ["bass","vocals"],
        ["bass","other"],

        ["vocals","other"],

        ["drums","bass","vocals"],
        ["drums","bass","other"],
        ["drums","vocals","other"],
        ["bass","vocals","other"]
    ]

    rand_mix = random.sample(STEMS, random.randint(2,4))
    mixes.append(rand_mix)

    return mixes


def mix_stems(stems_dict, selected):

    mix = np.zeros(SAMPLES, dtype=np.float32)

    for stem in selected:

        path = stems_dict[stem]

        if os.path.exists(path):

            y = load_audio(path)

            gain = np.random.uniform(0.6,1.4)

            mix += gain * y

    mix = mix / (np.max(np.abs(mix)) + 1e-6)

    return mix


def audio_to_mel(audio):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=2048,
        hop_length=512,
        n_mels=N_MELS,
        fmin=10,
        fmax=SR//2
    )

    mel = librosa.power_to_db(mel)
    mel = (mel - mel.mean())/(mel.std()+1e-6)

    return mel.astype(np.float32)


In [8]:
# PREPROCESS TRAIN

train_X = []
train_y = []

print("Preprocessing training data...")

for stems,label in tqdm(zip(train_songs,train_labels),total=len(train_songs)):

    for _ in range(TRAIN_MULTIPLIER):

        combos = generate_mixes()
        selected = random.choice(combos)

        audio = mix_stems(stems,selected)
        mel = audio_to_mel(audio)

        train_X.append(mel)
        train_y.append(label)

train_X = np.array(train_X)
train_y = np.array(train_y)

# PREPROCESS VAL

val_X = []
val_y = []

print("Preprocessing validation data...")

for stems,label in tqdm(zip(val_songs,val_labels),total=len(val_songs)):

    audio = mix_stems(stems,STEMS)
    mel = audio_to_mel(audio)

    val_X.append(mel)
    val_y.append(label)

val_X = np.array(val_X)
val_y = np.array(val_y)


Preprocessing training data...


100%|██████████| 900/900 [43:34<00:00,  2.91s/it]


Preprocessing validation data...


100%|██████████| 100/100 [00:47<00:00,  2.12it/s]


In [9]:
class MelDataset(Dataset):

    def __init__(self,X,y,augment=False):

        self.X = X
        self.y = y
        self.augment = augment

    def specaugment(self,x):

        if random.random() < 0.5:
            f = random.randint(8,32)
            f0 = random.randint(0,x.shape[1]-f)
            x[:,f0:f0+f,:] = 0

        if random.random() < 0.5:
            t = random.randint(10,40)
            t0 = random.randint(0,x.shape[2]-t)
            x[:,:,t0:t0+t] = 0

        return x

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):

        x = torch.tensor(self.X[idx]).unsqueeze(0)

        if self.augment:
            x = self.specaugment(x)

        # convert to 3 channels
        x = x.repeat(3,1,1)

        # resize for EfficientNet
        x = F.interpolate(
            x.unsqueeze(0),
            size=(224,224),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        y = torch.tensor(self.y[idx])

        return x,y


train_dataset = MelDataset(train_X,train_y,augment=True)
val_dataset = MelDataset(val_X,val_y,augment=False)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=2)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,num_workers=2)

In [10]:
# MODEL

model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    in_chans=3,
    num_classes=len(genres)
).to(DEVICE)

wandb.watch(model, log="all", log_freq=100)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# VALIDATION

def validate():

    model.eval()

    preds=[]
    targets=[]

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(DEVICE)
            y=y.to(DEVICE)

            out=model(x)

            pred=torch.argmax(out,1)

            preds.extend(pred.cpu().numpy())
            targets.extend(y.cpu().numpy())

    return f1_score(targets,preds,average="macro")


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [11]:
# TRAINING

best_f1 = 0

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x, y in train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    train_loss = total_loss / len(train_loader)
    val_f1 = validate()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss)
    print("Val F1:", val_f1)

    # ✅ log metrics to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_f1": val_f1,
        "lr": scheduler.get_last_lr()[0]
    })

    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(model.state_dict(), "best_model.pth")

        # ✅ save model to wandb
        wandb.save("best_model.pth")

        print("Model saved")

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



Epoch 1/25
Train Loss: 1.3198650487263996
Val F1: 0.7687276904382168
Model saved

Epoch 2/25
Train Loss: 0.8151957454284032
Val F1: 0.7714373617139804
Model saved

Epoch 3/25
Train Loss: 0.6782652580738068
Val F1: 0.8410989729225022
Model saved

Epoch 4/25
Train Loss: 0.6203283075491587
Val F1: 0.8288095238095238

Epoch 5/25
Train Loss: 0.5828958040475846
Val F1: 0.8365471650139842

Epoch 6/25
Train Loss: 0.5594657427072525
Val F1: 0.815953118089341

Epoch 7/25
Train Loss: 0.5451130352417628
Val F1: 0.8788942052099946
Model saved

Epoch 8/25
Train Loss: 0.5386309850215912
Val F1: 0.8303517260173605

Epoch 9/25
Train Loss: 0.533824578722318
Val F1: 0.7859515455304928

Epoch 10/25
Train Loss: 0.5258525270223617
Val F1: 0.8065738209159262

Epoch 11/25
Train Loss: 0.5199179774522782
Val F1: 0.8262497151970838

Epoch 12/25
Train Loss: 0.520145284930865
Val F1: 0.8167957773220931

Epoch 13/25
Train Loss: 0.5144626742601395
Val F1: 0.8510397205134048

Epoch 14/25
Train Loss: 0.51157577594121

In [12]:
wandb.summary["best_val_f1"] = best_f1

In [13]:
class TestDataset(Dataset):

    def __init__(self, path):
        self.files = sorted(os.listdir(path))
        self.path = path

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        name = self.files[idx]
        file_path = os.path.join(self.path, name)

        y,_ = librosa.load(file_path, sr=SR)

        segments=[]

        for start in range(0,len(y),SAMPLES):

            seg=y[start:start+SAMPLES]

            if len(seg)<SAMPLES:
                seg=np.pad(seg,(0,SAMPLES-len(seg)))

            mel=audio_to_mel(seg)

            mel=torch.from_numpy(mel).unsqueeze(0)

            mel=mel.repeat(3,1,1)

            mel=F.interpolate(
                mel.unsqueeze(0),
                size=(224,224),
                mode="bilinear",
                align_corners=False
            ).squeeze(0)

            segments.append(mel)

        segments=torch.stack(segments)

        return segments,name


In [14]:
# INFERENCE

model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

test_dataset = TestDataset(TEST_PATH)
test_loader = DataLoader(test_dataset,batch_size=1,shuffle=False)

preds=[]
names=[]

with torch.no_grad():

    for x,n in test_loader:

        x=x.to(DEVICE)

        B,S,C,H,W=x.shape

        x=x.view(B*S,C,H,W)

        outputs=model(x)

        outputs=outputs.view(B,S,-1).mean(1)

        preds.extend(torch.argmax(outputs,1).cpu().numpy())
        names.extend(n)

# SAVE SUBMISSION

genre_preds=[idx_to_label[p] for p in preds]

submission=pd.read_csv(SUB_PATH)
submission["genre"]=genre_preds
submission.to_csv("dl_Submission.csv",index=False)

print("Submission.csv saved ✅")

Submission.csv saved ✅


In [15]:
wandb.finish()

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train_loss,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_f1,▁▁▆▅▅▄█▅▂▃▅▄▆▃▃▇▆▆▇▇▆▇▆▆▅
best_val_f1,0.87889
epoch,25
lr,0
train_loss,0.50404
val_f1,0.83637
